In [ ]:
import numpy as np
import torch
import pandas as pd
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
import os
import sklearn
import io
import seaborn as sns
from sklearn.utils import resample
import scipy as sp
import scipy.stats as st



## Global parameters for plots
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

from torch.autograd import Variable
torch. __version__


sample_space0 = torch.linspace(1,10, steps = 10)
sample_space1 = torch.linspace(20,100, steps = 9)
sample_space2 = torch.linspace(100,1000, steps = 10)

sample_space = torch.cat((sample_space1,sample_space2),dim = 0)
sample_space = sample_space.unique()
print(sample_space)
N_samples = sample_space.size(0)
####################### Model parameters#######################


In [ ]:
performance_limit = 0.75
param_coef = -0.75
param_exp = -0.57
c_1 = 0.4
zeta = 0.85
pre_det_threshold = torch.tensor([0.90,0.89,0.88,0.87,0.86,0.85,0.84,0.83,0.82,0.81,0.80,0.79,0.78,0.77,0.76,0.75,0.74,0.73])


anchored_samples_generated = 100


##################################Exact-Model########################################
y_exact = performance_limit + param_coef*(sample_space**param_exp)
y_exact_OF = y_exact + zeta*(sample_space**(-0.5))


##################################Function-Data-Generator########################################
def data_generator(N_samples,sample_space,performance_limit,param_coef,param_exp,c_1,zeta,anchored_samples_generated,pre_det_threshold):
    #y_L = 0.78
    torch.manual_seed(42)

    average_accuracy_experiment = torch.zeros(N_samples,1)
    N_accepted = torch.zeros(N_samples,1)
    #################Data-Generator-Fixed-Number-Classifiers-per-Anchor-Size#######################
    number_iteration = 1

    experiment_accuracy_all = torch.empty(0)

    empirical_mean_all_obs = torch.zeros(N_samples,1)
    lb_original_data = torch.zeros(N_samples,1)
    ub_original_data = torch.zeros(N_samples,1)

    experiment_bias_dataset = torch.empty(0)
    censored_point_dataset = torch.empty(0)
    N_dataset = torch.empty(0)
    N_dataset_censored_plot = torch.empty(0)
    average_bias = torch.empty(0)
    y_L_all = torch.empty(0)
    for j in range(N_samples):
        sample_interest = sample_space[j]
        accuracy_generator_mean = performance_limit + (param_coef*(sample_interest**(param_exp)))
        accuracy_generator_sigma = (c_1)/np.sqrt(sample_interest)
        noise_mean = zeta*(sample_interest**(-0.5))
        POOL_empirical_learning_curve = accuracy_generator_mean + accuracy_generator_sigma*torch.randn(anchored_samples_generated)

        POOL_experiment_accuracy = POOL_empirical_learning_curve + noise_mean
        POOL_experiment_accuracy = torch.clip(POOL_experiment_accuracy,min = 0.5,max = 1.0)

        pre_threshold = pre_det_threshold[j]


        pool_sorted,sorted_indices = torch.sort(POOL_experiment_accuracy,descending = True)

        experiment_bias_anchored = pool_sorted[(pool_sorted>=pre_threshold)]
        experiment_bias_dataset = torch.cat([experiment_bias_dataset, experiment_bias_anchored])

        experiment_bias_censored = pool_sorted[(pool_sorted<pre_threshold)]
        censored_point_dataset  = torch.cat([censored_point_dataset, experiment_bias_censored])

        average_bias_local = torch.mean(experiment_bias_anchored,dim = 0)
        average_bias = torch.cat([average_bias,average_bias_local.reshape(1)])
        size_accepted_studies = experiment_bias_anchored.size()
        size_rejected_studies = experiment_bias_censored.size()


        n_dataset_accepted_local = sample_interest.repeat(size_accepted_studies,1)

        N_dataset = torch.cat([N_dataset,n_dataset_accepted_local])

        n_dataset_rejected_local = sample_interest.repeat(size_rejected_studies,1)

        N_dataset_censored_plot = torch.cat([N_dataset_censored_plot,n_dataset_rejected_local])

        y_L_repeat = pre_threshold.repeat(size_accepted_studies,1)
        y_L_all = torch.cat([y_L_all,y_L_repeat])

        confidence_bounds_original_data = [np.percentile(POOL_experiment_accuracy, 2.5),np.percentile(POOL_experiment_accuracy, 97.5)]
        lb_original_data[j] =  confidence_bounds_original_data[0]
        ub_original_data[j] = confidence_bounds_original_data[1]
        empirical_mean_all_obs[j] = torch.mean(POOL_experiment_accuracy,dim = 0)

########################################The-End##############################################
    N_dataset = N_dataset.view(-1,1)

    N_dataset_censored_plot = N_dataset_censored_plot.view(-1,1)


    experiment_bias = experiment_bias_dataset.view(-1,1)

    censored_data = censored_point_dataset.view(-1,1)
    y_L_all = y_L_all.view(-1,1)
    True_learning_curve = performance_limit + param_coef*(sample_space**(param_exp))
    True_learning_curve_OF = performance_limit + param_coef*(sample_space**(param_exp)) + zeta*(sample_space**(-0.5))
    empirical_mean_all_obs = empirical_mean_all_obs.view(-1,)
    return (experiment_bias,empirical_mean_all_obs,average_bias,N_dataset,lb_original_data,ub_original_data,censored_data,N_dataset_censored_plot,True_learning_curve,True_learning_curve_OF,y_L_all)
########################################End-of-Functon##############################################
experiment_bias,empirical_mean_all_obs,average_bias,N_dataset,lb_original_data,ub_original_data,censored_data,N_dataset_censored_plot,True_learning_curve,True_learning_curve_OF,y_L_all = data_generator(N_samples,sample_space,performance_limit,param_coef,param_exp,c_1,zeta,anchored_samples_generated,pre_det_threshold)
N_dataset = N_dataset.numpy()
experiment_bias = experiment_bias.numpy()
experiment_bias = experiment_bias.squeeze()
N_dataset = N_dataset.squeeze()
y_L_all = y_L_all.numpy().squeeze()
pre_det_threshold = pre_det_threshold.numpy().squeeze()
############################################
############################################
N_dataset_eval,N_dataset_rank = np.unique(N_dataset,return_index=True)



In [ ]:
############################################
N_dataset_eval,N_dataset_rank = np.unique(N_dataset,return_index=True)

from matplotlib.legend_handler import HandlerTuple
sns.set_style("white")
sns.set_style("ticks")

fig = plt.figure(figsize=(3.42,2.36), dpi = 200)
ax2 = plt.axes()
scatter1 = ax2.scatter(N_dataset,experiment_bias,color = 'olivedrab',s=0.5,linewidth = 1)
scatter2 = ax2.scatter(N_dataset_censored_plot,censored_data,color = 'mistyrose',s=0.5,linewidth = 1)

pl4, = ax2.plot(N_dataset_eval,empirical_mean_all_obs,linestyle = 'dashdot',color = 'navy',linewidth=1,markersize=2,label = 'True learning curve')
ax2.fill_between(N_dataset_eval,lb_original_data.numpy().squeeze(),ub_original_data.numpy().squeeze(),color = 'lightsteelblue',alpha = 0.1)


pl6, = ax2.plot(sample_space,True_learning_curve,linestyle = 'dashed',color = 'goldenrod',linewidth=1,markersize=1,label = 'True learning curve')

ax2.set_xscale('log')



#ax2.set_xscale('log')

ax2.tick_params(axis='y',which='major',direction='out',length=1,width=0.5,color='black',pad=2,labelsize=6,labelcolor='black')
ax2.tick_params(axis='x',which='minor',direction='out',length=1,width=0.5,color='black',labelcolor='black')
ax2.tick_params(axis='x',which='major',direction='out',length=2,width=1,color='black',pad=2,labelsize=6,labelcolor='black')
ax2.set_yticks([0.5,0.6, 0.70, 0.80,0.9,1.0])
ax2.set_ylim(0.4,1)
ax2.set_xlim(10,1000)
plt.xticks(weight = 'bold')
plt.yticks(weight = 'bold')
plt.xlabel('Sample Size',fontsize = 6, fontweight='bold')
plt.ylabel('Accuracy',fontsize = 6, fontweight='bold')

ax2.grid(axis='x', color='0.9')
ax2.grid(axis='y', color='0.9')

legend1 = ax2.legend([scatter1,scatter2,pl4,pl6], ['Overoptimistic accuracies','Truncated samples','Empirical mean - all accuracies',r'True learning curve - $A + \alpha N^{\beta}$'],
           scatterpoints=1, numpoints=1,fontsize = 5,loc = 'lower right',handler_map={tuple: HandlerTuple(ndivide=None)})

plt.savefig('/results/Published_truncated_accuracies.png', format='png',dpi = 200,bbox_inches = 'tight')

plt.show()



In [ ]:
def estimate_threshold(sample_used,accuracy_used):
  size_Sample_space = sample_used.shape[0]
  Sample_space_unique = np.unique(sample_used)
  size_Sample_space_unique = Sample_space_unique.shape[0]

  threshold_estimated = np.zeros(size_Sample_space)
  stride = 1
  old_scale = 100
  for window_start_point in range(0,size_Sample_space_unique,1):
    if (window_start_point == size_Sample_space_unique - 1):
      window = window_start_point
    else:
      window = np.linspace(window_start_point,window_start_point+stride-1,stride).astype(int)

    window_on_samples = Sample_space_unique[window]
    Sample_space_indexes = np.where(np.isin(sample_used,window_on_samples))[0]
    Reported_accuracy_quant = accuracy_used[Sample_space_indexes]
    new_scale = np.min(Reported_accuracy_quant)
    final_scale = np.minimum(old_scale,new_scale)
    threshold_estimated[Sample_space_indexes] = final_scale
    old_scale = final_scale

  y_L = threshold_estimated
  return y_L

def estimate_threshold_evaluation(y_L,sample_used,N_dataset_eval):
  sample_used_eval,sample_used_rank = np.unique(sample_used,return_index=True)
  y_L_eval = y_L[sample_used_rank]

  N_samples_eval = N_dataset_eval.size

  size_Sample_space_unique = sample_used_eval.shape[0]

  threshold_eval_estimated = np.zeros(N_samples_eval)
  locations = np.where(np.isin(N_dataset_eval,sample_used_eval))[0]
  threshold_eval_estimated[locations] = y_L_eval

  for i in range(len(threshold_eval_estimated)):

    if threshold_eval_estimated[i] == 0:
      left = i - 1
      right = i + 1

      while left >= 0 and threshold_eval_estimated[left] == 0:
        left -= 1

      while right < len(threshold_eval_estimated) and threshold_eval_estimated[right] == 0:
        right += 1

      threshold_eval_estimated[i] = threshold_eval_estimated[left] if left >= 0 else threshold_eval_estimated[right]


  y_L_eval = threshold_eval_estimated

  return y_L_eval

In [ ]:
from pymoo.core.problem import ElementwiseProblem

class MyProblem(ElementwiseProblem):

    def __init__(self):
        super().__init__(n_var=5,
                         n_obj=2,
                         n_ieq_constr=0,
                         xl=np.array([0.5,-2,-1,0,0.001]),
                         xu=np.array([1,-0.5,-0.05,1,0.5]))

    def _evaluate(self, x, out, *args, **kwargs):
      mu_unique = x[0] + x[1]*(sample_used_new_unique**x[2]) + x[3]*(sample_used_new_unique**(-0.5))
      zprime = np.sqrt(sample_used_new_unique)*(y_L_used_unique - mu_unique)/x[4]
      if ((1-sp.stats.norm.cdf(zprime,loc = 0,scale = 1)).all() ==False):
        epsilon = 10**-20
      else:
        epsilon = 0
      epsi_prime = (sp.stats.norm.pdf(zprime,loc = 0,scale = 1))/(1-sp.stats.norm.cdf(zprime,loc = 0,scale = 1)+epsilon)

      f1 = sum(((mu_unique + (x[4]/np.sqrt(sample_used_new_unique))* epsi_prime) - mean_used)**2)

      f2 = sum((((x[4]**2)/sample_used_new_unique)*(1 + (zprime*epsi_prime) - epsi_prime) - variance_used)**2)

      out["F"] = [f1,f2]

problem = MyProblem()





In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.visualization.scatter import Scatter

algorithm = NSGA2(
    pop_size=40,
    n_offsprings=10,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(eta=20),
    eliminate_duplicates=True)




In [ ]:
### We create 10000 bootstrap samples for computing the confidence intervals. To run the code faster, we can change the n_repeats to 100.

from pymoo.termination import get_termination
termination = get_termination("n_gen", 500)

from pymoo.optimize import minimize
from pymoo.decomposition.asf import ASF


N_samples = N_dataset.size
N_dataset_eval,N_dataset_rank = np.unique(N_dataset,return_index=True)
N_samples_eval = N_dataset_eval.size
n_repeats = 100
itr_for_initialization = 1

model_recovered_ML = np.zeros((n_repeats,N_samples_eval))
model_recovered_ML_OF = np.zeros((n_repeats,N_samples_eval))

MLR_estimated = np.zeros((n_repeats,N_samples_eval))
target_acc_vec = np.zeros((n_repeats,1))
alpha_star_vec = np.zeros((n_repeats,1))
beta_star_vec = np.zeros((n_repeats,1))
zeta_star_vec = np.zeros((n_repeats,1))
c1_vec = np.zeros((n_repeats,1))

n_samples_used=len(N_dataset)


def eval_func_complete(target_accuracy,alpha_star,beta_star,zeta_star,c_1_star):

  mu = target_accuracy + alpha_star*(N_dataset_eval**beta_star) + zeta_star*(N_dataset_eval**(-0.5))
  z = np.sqrt(N_dataset_eval)*(y_L_eval - mu)/c_1_star
  if ((1-sp.stats.norm.cdf(z,loc = 0,scale = 1)).all() ==False):
    epsilon = 10**-20
  else:
    epsilon = 0
  return mu + (c_1_star/np.sqrt(N_dataset_eval))* (sp.stats.norm.pdf(z,loc = 0,scale = 1))/(1-sp.stats.norm.cdf(z,loc = 0,scale = 1)+epsilon)

def caluclate_variance(sample_used_new,accuracy_used_new):
  v_sample_used_unique = np.unique(sample_used_new)
  variance_used = []
  mean_used = []
  for N_S in (v_sample_used_unique):
    v_sample_index = np.where(np.isin(sample_used_new,N_S))[0]
    A_temp = []
    A_temp = accuracy_used_new[v_sample_index]
    A_temp = np.array([A_temp])
    variance_cal = np.var(A_temp)
    mean_cal = np.mean(A_temp)
    variance_used.append(variance_cal)
    mean_used.append(mean_cal)
  variance_used = np.array([variance_used])
  mean_used = np.array([mean_used])

  return variance_used.squeeze(),mean_used.squeeze()




prog_bar1 = tqdm(total = n_repeats)
for i in range(n_repeats):
  time.sleep(0.3)
  n_samples_used=len(N_dataset)

  [sample_used,accuracy_used] = resample(N_dataset,experiment_bias,n_samples = len(N_dataset))

  ########## Mean Acc - Required for Initialization##################
  sorting_index = np.argsort(sample_used)   #argsort sort in ascending order
  sample_used_new = sample_used[sorting_index]
  sample_used_new_unique,unique_samples_indices = np.unique(sample_used_new, return_index=True) #unique sort in ascending order in this case doesn't have any effect
  accuracy_used_new = accuracy_used[sorting_index]
  sample_used_unique = np.unique(sample_used)

  ##########End of Block - Mean Acc##################
  y_L_used = estimate_threshold(sample_used_new,accuracy_used_new)
  y_L_eval = estimate_threshold_evaluation(y_L_used,sample_used_new,N_dataset_eval)
  y_L_used_unique = y_L_used[unique_samples_indices] #This is for variance calculations in the objective function

  variance_used,mean_used = caluclate_variance(sample_used_new,accuracy_used_new)


  res = minimize(problem,
               algorithm,
               termination,
               seed=1,
               save_history=True,
               verbose=False)

  X = res.X
  F = res.F


  approx_ideal = F.min(axis=0)
  approx_nadir = F.max(axis=0)
  nF = (F - approx_ideal) / (approx_nadir - approx_ideal)

  weights = np.array([0.999999, 0.000001])
  decomp = ASF()
  best_answer = decomp.do(nF, 1/weights).argmin()
  X_final = X[best_answer,:]
  target_accuracy = X_final[0]
  alpha_star = X_final[1]
  beta_star = X_final[2]
  zeta_star = X_final[3]
  c_1_star = X_final[4]


  target_acc_vec[i,0] = target_accuracy
  alpha_star_vec[i,0] = alpha_star
  beta_star_vec[i,0] = beta_star
  zeta_star_vec[i,0] = zeta_star
  c1_vec[i,0] = c_1_star

  model_recovered_ML[i,:] = target_accuracy + alpha_star*(N_dataset_eval**beta_star)
  model_recovered_ML_OF[i,:] = target_accuracy + alpha_star*(N_dataset_eval**beta_star)+zeta_star*(N_dataset_eval**(-0.5))

  MLR_estimated[i,:] = eval_func_complete(target_accuracy,alpha_star,beta_star,zeta_star,c_1_star)

  prog_bar1.update(1)
prog_bar1.close()



In [ ]:
MLR_temp = MLR_estimated
confidence_bounds_set_MLR = []
for cb in range(N_samples_eval):
  confidence_bounds_MLR = [np.percentile(MLR_temp[:,cb], 2.5),np.percentile(MLR_temp[:,cb], 97.5)]

  confidence_bounds_set_MLR.append(confidence_bounds_MLR)

lb_MLR = np.array(confidence_bounds_set_MLR)[:,0]
ub_MLR = np.array(confidence_bounds_set_MLR)[:,1]

MLR_estimated_avg = np.mean(MLR_estimated,0)


B_temp = model_recovered_ML
confidence_bounds_set_mr = []
for cb in range(N_samples_eval):
  confidence_bounds_mr = [np.percentile(B_temp[:,cb], 2.5),np.percentile(B_temp[:,cb], 97.5)]

  confidence_bounds_set_mr.append(confidence_bounds_mr)
lb_mr = np.array(confidence_bounds_set_mr)[:,0]
ub_mr = np.array(confidence_bounds_set_mr)[:,1]

model_recovered_mr = np.mean(model_recovered_ML,0)
C_temp = model_recovered_ML_OF
confidence_bounds_set_OF = []
for cb in range(N_samples_eval):
  confidence_bounds_OF = [np.percentile(C_temp[:,cb], 2.5),np.percentile(C_temp[:,cb], 97.5)]


  confidence_bounds_set_OF.append(confidence_bounds_OF)
lb_OF = np.array(confidence_bounds_set_OF)[:,0]
ub_OF = np.array(confidence_bounds_set_OF)[:,1]

model_recovered_OF = np.mean(model_recovered_ML_OF,0)





In [ ]:
from matplotlib.legend_handler import HandlerTuple

sns.set_style("white")
sns.set_style("ticks")
fig = plt.figure(figsize=(3.42,2.36), dpi = 200)
ax2 = plt.axes()

scatter1 = ax2.scatter(N_dataset,experiment_bias,color = 'lightsteelblue',s=0.5,linewidth = 1)

pl1, = ax2.plot(N_dataset_eval,MLR_estimated_avg,'-',color = 'royalblue',linewidth=1,markersize=1,label = "Fit to the Mill's ratio")



ax2.fill_between(N_dataset_eval,lb_MLR,ub_MLR,color ='navy',alpha = 0.1)

pl3, = ax2.plot(N_dataset_eval,model_recovered_mr,linestyle = 'dashdot',color = 'red',linewidth=1,markersize=2,label = 'Estimated learning curve')

ax2.fill_between(N_dataset_eval,lb_mr,ub_mr,color = 'tomato',alpha = 0.1)

pl5, = ax2.plot(N_dataset_eval,y_exact,linestyle = 'dashed',color = 'olivedrab',linewidth=1,markersize=1,label = 'True learning curve')

ax2.set_xscale('log')

ax2.tick_params(axis='y',which='major',direction='out',length=1,width=0.5,color='black',pad=2,labelsize=6,labelcolor='black')
ax2.tick_params(axis='x',which='minor',direction='out',length=1,width=0.5,color='black',labelcolor='black')
ax2.tick_params(axis='x',which='major',direction='out',length=2,width=1,color='black',pad=2,labelsize=6,labelcolor='black')
ax2.set_yticks([0.5,0.6, 0.7, 0.8, 0.9, 1.0])
ax2.set_ylim(0.45,1.0)
ax2.set_xlim(10,1000)
plt.xticks(weight = 'bold')
plt.yticks(weight = 'bold')
plt.xlabel('Sample Size',fontsize = 6, fontweight='bold')
plt.ylabel('Accuracy',fontsize = 6, fontweight='bold')


legend1 = ax2.legend([scatter1,pl1,pl3,pl5], ['Overoptimistic accuracies','Fit to overoptimistic accuracies','Estimated learning curve','True learning curve'],
          scatterpoints=1, numpoints=1,fontsize = 6,loc = 'lower right',handler_map={tuple: HandlerTuple(ndivide=None)})

ax2.grid(axis='x', color='0.9')
ax2.grid(axis='y', color='0.9')
plt.title('Experiment 1 - Problem 2',fontsize = 6, fontweight='bold')
plt.savefig('syn_data_final_result.png', format='png',dpi = 200,bbox_inches = 'tight')

plt.show()











